In [ ]:
import rclpy
from rclpy.node import Node
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
import threading
from ipywidgets import FloatSlider, Button, VBox, HBox, Label
from IPython.display import display

In [ ]:
class GripperController(Node):
    def __init__(self):
        super().__init__('gripper_controller')
        self.publisher = self.create_publisher(
            JointTrajectory, '/joint_trajectory_controller/joint_trajectory', 10)
        self.get_logger().info("GripperController started")

    def send_grasp_command(self, positions):
        traj = JointTrajectory()
        traj.joint_names = [
            'joint_lift',
            'joint_wrist_pitch',
            'joint_wrist_roll',
            'joint_wrist_yaw',
            'joint_head_pan',
            'joint_head_tilt',
            'joint_gripper_slide',
            'joint_arm_l0'
        ]

        point = JointTrajectoryPoint()
        point.positions = positions
        point.time_from_start.sec = 1  
        point.time_from_start.nanosec = 0

        traj.points.append(point)
        self.publisher.publish(traj)
        self.get_logger().info(f"Published trajectory: {positions}")

In [ ]:
def start_node():
    global node, executor_thread
    rclpy.init(args=None)
    node = GripperController()
    executor_thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
    executor_thread.start()
    print("Node running in background...")

start_node()

In [ ]:
# Define sliders for each joint
joint_sliders = {
    "joint_lift": FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, description="Lift"),
    "joint_wrist_pitch": FloatSlider(value=0.0, min=-1.5, max=1.5, step=0.01, description="Pitch"),
    "joint_wrist_roll": FloatSlider(value=-0.5, min=-1.5, max=1.5, step=0.01, description="Roll"),
    "joint_wrist_yaw": FloatSlider(value=0.0, min=-1.5, max=1.5, step=0.01, description="Yaw"),
    "joint_head_pan": FloatSlider(value=0.02, min=-1.0, max=1.0, step=0.01, description="Head Pan"),
    "joint_head_tilt": FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.01, description="Head Tilt"),
    "joint_gripper_slide": FloatSlider(value=-0.1, min=-0.5, max=0.5, step=0.01, description="Gripper Slide"),
    "joint_arm_l0": FloatSlider(value=0.02, min=0.0, max=0.13, step=0.01, description="Arm L0")
}

# Create "Send" button
send_button = Button(description="Send Trajectory", button_style='success')

def on_send_clicked(_):
    positions = [slider.value for slider in joint_sliders.values()]
    node.send_grasp_command(positions)

send_button.on_click(on_send_clicked)

# Create "Stop" button
stop_button = Button(description="Stop Node", button_style='danger')

def on_stop_clicked(_):
    print("Shutting down node...")
    node.destroy_node()
    rclpy.shutdown()

stop_button.on_click(on_stop_clicked)

# Display UI
controls = VBox(list(joint_sliders.values()) + [HBox([send_button, stop_button])])
display(controls)